# AIDCE-Sim v1 — robust Colab, Linux, and Windows launcher

**AIDCE-Sim** is an interactive, data-free Python simulator for studying the energy-use patterns of AI data centres.

**Developer:** Partha Pratim Ray, Sikkim University, India  
**Email:** parthapratimray1986@gmail.com  
**Notebook revision:** 2 August 2026

This notebook is designed to be safely rerun. It:

- finds the actual extracted project directory instead of relying on `%cd`;
- handles Colab's preinstalled `blinker` conflict;
- stops older Streamlit and Cloudflared processes before restarting;
- never downloads directly over a running Cloudflared executable;
- checks Streamlit's local health endpoint before opening a tunnel;
- obtains the Quick Tunnel hostname from Cloudflared's metrics endpoint;
- verifies the public URL before displaying it;
- retries with HTTP/2 when the default tunnel protocol fails.

> **Important:** Cloudflare Quick Tunnels are temporary development links and have no uptime guarantee. For a stable public deployment, use Streamlit Community Cloud or a named Cloudflare Tunnel.

In [13]:
# Cell 1 — Runtime configuration and common imports

from pathlib import Path
import importlib.util
import json
import os
import platform
import re
import shutil
import signal
import socket
import subprocess
import sys
import time
import zipfile

# Detect whether this notebook is running in Google Colab.
IS_COLAB = "google.colab" in sys.modules

# Choose how AIDCE-Sim is obtained.
# Keep ZIP mode for the current source archive.
# After publishing the package, select TEST_PYPI.
INSTALL_MODE = "ZIP"  # @param ["ZIP", "TEST_PYPI"]

# Current ZIP settings.
ZIP_FILENAME = "AIDCE_Sim_v1.zip"  # @param {type:"string"}

# Future TestPyPI settings. These assume the package import name is aidce_sim
# and that aidce_sim/app.py is included in the distribution.
TESTPYPI_PROJECT_NAME = "aidce-sim"  # @param {type:"string"}
TESTPYPI_IMPORT_NAME = "aidce_sim"  # @param {type:"string"}
TESTPYPI_VERSION = ""  # @param {type:"string"}

# Server settings.
PREFERRED_PORT = 8501  # @param {type:"integer"}

# Colab uses /content. A local notebook uses its current directory.
BASE_DIR = Path("/content") if IS_COLAB else Path.cwd()
WORKSPACE_DIR = BASE_DIR / "aidce_workspace"
STATE_DIR = BASE_DIR / ".aidce_runtime"
STATE_DIR.mkdir(parents=True, exist_ok=True)

print("Environment:", "Google Colab" if IS_COLAB else platform.system())
print("Python:", sys.version.split()[0])
print("Base directory:", BASE_DIR)

Environment: Google Colab
Python: 3.12.13
Base directory: /content


In [14]:
# Cell 2 — Upload/extract the ZIP, or install from TestPyPI


def run_checked(command, *, cwd=None):
    """Run a command and stop immediately if it fails."""
    print("Running:", " ".join(map(str, command)))
    subprocess.run(list(map(str, command)), cwd=cwd, check=True)


if INSTALL_MODE == "ZIP":
    zip_path = BASE_DIR / ZIP_FILENAME

    # In Colab, request the ZIP only when it is not already available.
    if IS_COLAB and not zip_path.exists():
        from google.colab import files

        print(f"Upload {ZIP_FILENAME}")
        uploaded = files.upload()
        uploaded_names = list(uploaded)
        if not uploaded_names:
            raise FileNotFoundError("No ZIP file was uploaded.")

        # Rename the uploaded archive to the configured filename when necessary.
        uploaded_path = BASE_DIR / uploaded_names[0]
        if uploaded_path != zip_path:
            uploaded_path.replace(zip_path)

    if not zip_path.exists():
        raise FileNotFoundError(
            f"Could not find {zip_path}. Put the ZIP beside this notebook "
            "or update ZIP_FILENAME in Cell 1."
        )

    WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as archive:
        archive.extractall(WORKSPACE_DIR)

    # Find a folder containing both app.py and requirements.txt.
    candidates = sorted(
        {
            path.parent.resolve()
            for path in WORKSPACE_DIR.rglob("app.py")
            if (path.parent / "requirements.txt").exists()
        }
    )
    if not candidates:
        raise FileNotFoundError(
            "The extracted archive does not contain app.py and requirements.txt "
            "in the same project directory."
        )

    # Prefer the expected AIDCE_Sim folder when several candidates exist.
    PROJECT_DIR = next(
        (folder for folder in candidates if folder.name.lower() == "aidce_sim"),
        candidates[0],
    )
    APP_PATH = PROJECT_DIR / "app.py"

else:
    package_spec = TESTPYPI_PROJECT_NAME
    if TESTPYPI_VERSION.strip():
        package_spec += f"=={TESTPYPI_VERSION.strip()}"

    run_checked(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--index-url",
            "https://test.pypi.org/simple/",
            "--extra-index-url",
            "https://pypi.org/simple/",
            package_spec,
        ]
    )

    spec = importlib.util.find_spec(TESTPYPI_IMPORT_NAME)
    if spec is None or spec.origin is None:
        raise ImportError(
            f"Installed {TESTPYPI_PROJECT_NAME}, but import package "
            f"{TESTPYPI_IMPORT_NAME!r} was not found."
        )

    PROJECT_DIR = Path(spec.origin).resolve().parent
    APP_PATH = PROJECT_DIR / "app.py"
    if not APP_PATH.exists():
        raise FileNotFoundError(
            f"Expected packaged Streamlit app at {APP_PATH}. "
            "Include aidce_sim/app.py in the TestPyPI distribution."
        )

# Do not rely on notebook-specific %cd magic. This works in Colab, Jupyter,
# Linux, macOS, and Windows.
os.chdir(PROJECT_DIR)

print("Project directory:", PROJECT_DIR)
print("Application:", APP_PATH)
print("Project files:")
for item in sorted(PROJECT_DIR.iterdir()):
    print(" -", item.name)

Project directory: /content/aidce_workspace/AIDCE_Sim
Application: /content/aidce_workspace/AIDCE_Sim/app.py
Project files:
 - .gitignore
 - README.md
 - __pycache__
 - app.py
 - article_blueprint.md
 - example_config.json
 - forecasting.py
 - methodology_notes.md
 - requirements.txt
 - sample_forecast.csv
 - sample_forecast_metrics.csv
 - sample_kpis.csv
 - sample_preview.png
 - sample_scenario_summary.csv
 - sample_timeseries.csv
 - simulator.py
 - smoke_test.py


In [15]:
# Cell 3 — Install and verify Python dependencies


def pip_install(*arguments):
    command = [sys.executable, "-m", "pip", "install", *arguments]
    print("Running:", " ".join(command))
    subprocess.run(command, check=True)


# Colab may contain blinker 1.4 installed through distutils. Installing a newer
# copy without first uninstalling the system copy avoids the known pip error.
if IS_COLAB:
    pip_install("-q", "--ignore-installed", "blinker>=1.9,<2")

# ZIP mode needs the source project's dependency file. TestPyPI normally
# installs declared dependencies automatically, but utility packages are still
# installed below for process and health management.
requirements_path = PROJECT_DIR / "requirements.txt"
if INSTALL_MODE == "ZIP":
    if not requirements_path.exists():
        raise FileNotFoundError(requirements_path)
    pip_install(
        "-q",
        "--upgrade-strategy",
        "only-if-needed",
        "-r",
        str(requirements_path),
    )

# Utilities used only by this launcher notebook.
pip_install("-q", "--upgrade-strategy", "only-if-needed", "psutil", "requests")

# Import only after installation is complete.
import blinker
import numpy
import pandas
import plotly
import psutil
import requests
import sklearn
import statsmodels
import streamlit

print("Blinker:", getattr(blinker, "__version__", "installed"))
print("Streamlit:", streamlit.__version__)
print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("scikit-learn:", sklearn.__version__)
print("statsmodels:", statsmodels.__version__)
print("Plotly:", plotly.__version__)
print("Dependency installation successful.")

Running: /usr/bin/python3 -m pip install -q --ignore-installed blinker>=1.9,<2
Running: /usr/bin/python3 -m pip install -q --upgrade-strategy only-if-needed -r /content/aidce_workspace/AIDCE_Sim/requirements.txt
Running: /usr/bin/python3 -m pip install -q --upgrade-strategy only-if-needed psutil requests
Blinker: installed
Streamlit: 1.60.0
NumPy: 2.0.2
Pandas: 2.2.2
scikit-learn: 1.6.1
statsmodels: 0.14.6
Plotly: 5.24.1
Dependency installation successful.


In [16]:
# Cell 4 — Run the simulation smoke test

smoke_test_path = PROJECT_DIR / "smoke_test.py"

if smoke_test_path.exists():
    subprocess.run(
        [sys.executable, str(smoke_test_path)],
        cwd=PROJECT_DIR,
        check=True,
    )
else:
    # Future TestPyPI fallback when smoke_test.py is not distributed.
    sys.path.insert(0, str(PROJECT_DIR.parent))
    simulator_module = (
        f"{TESTPYPI_IMPORT_NAME}.simulator"
        if INSTALL_MODE == "TEST_PYPI"
        else "simulator"
    )
    forecasting_module = (
        f"{TESTPYPI_IMPORT_NAME}.forecasting"
        if INSTALL_MODE == "TEST_PYPI"
        else "forecasting"
    )

    simulator = __import__(simulator_module, fromlist=["*"])
    forecasting = __import__(forecasting_module, fromlist=["*"])

    cfg = simulator.SimulationConfig(
        days=2,
        freq_minutes=30,
        gpus=256,
        grid_capacity_mw=2.0,
    )
    frame = simulator.simulate_ai_data_center(cfg)
    assert len(frame) == 96
    assert (frame["facility_power_mw"] >= frame["it_power_mw"]).all()
    forecast = forecasting.run_forecasting(
        frame["grid_power_mw"],
        cfg.freq_minutes,
        forecast_hours=6,
        holdout_hours=6,
    )
    assert len(forecast.future) == 12
    print("Packaged smoke test passed.")

In [24]:
# Cell 5 — Start Streamlit safely and verify its local health

import psutil
import requests

STREAMLIT_PID_FILE = STATE_DIR / "streamlit.pid"
TUNNEL_PID_FILE = STATE_DIR / "cloudflared.pid"
STREAMLIT_LOG = STATE_DIR / "streamlit.log"
TUNNEL_LOG = STATE_DIR / "cloudflared.log"


def terminate_process_tree(pid, timeout=8):
    """Terminate a process and its children without killing the notebook kernel."""
    try:
        process = psutil.Process(int(pid))
    except (psutil.NoSuchProcess, psutil.Error, ValueError):
        return

    children = process.children(recursive=True)
    for child in children:
        try:
            child.terminate()
        except psutil.Error:
            pass
    try:
        process.terminate()
    except psutil.Error:
        pass

    _, alive = psutil.wait_procs(children + [process], timeout=timeout)
    for remaining in alive:
        try:
            remaining.kill()
        except psutil.Error:
            pass



def stop_pid_file(pid_file):
    if not pid_file.exists():
        return
    try:
        pid = int(pid_file.read_text().strip())
        terminate_process_tree(pid)
    finally:
        pid_file.unlink(missing_ok=True)



def stop_legacy_colab_processes():
    """Clean processes created by the older notebook cells in this runtime."""
    if not IS_COLAB:
        return

    current_pid = os.getpid()
    for process in psutil.process_iter(["pid", "name", "cmdline"]):
        try:
            if process.info["pid"] == current_pid:
                continue
            command = " ".join(process.info.get("cmdline") or [])
            name = (process.info.get("name") or "").lower()
            is_old_tunnel = "cloudflared" in name or "/content/cloudflared" in command
            is_old_streamlit = "streamlit" in command and "app.py" in command
            if is_old_tunnel or is_old_streamlit:
                terminate_process_tree(process.info["pid"])
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass



def port_is_free(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        try:
            sock.bind(("127.0.0.1", int(port)))
            return True
        except OSError:
            return False



def choose_port(preferred):
    if port_is_free(preferred):
        return int(preferred)
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind(("127.0.0.1", 0))
        return int(sock.getsockname()[1])



def tail(path, lines=40):
    if not Path(path).exists():
        return ""
    return "\n".join(Path(path).read_text(errors="replace").splitlines()[-lines:])


# Stop both the new PID-managed processes and processes started by the older
# notebook. This makes the cell safe to rerun during the same Colab session.
stop_pid_file(TUNNEL_PID_FILE)
stop_pid_file(STREAMLIT_PID_FILE)
stop_legacy_colab_processes()

PORT = choose_port(PREFERRED_PORT)

streamlit_command = [
    sys.executable,
    "-m",
    "streamlit",
    "run",
    str(APP_PATH),
    "--server.port",
    str(PORT),
    "--server.address",
    "127.0.0.1",
    "--server.headless",
    "true",
    "--browser.gatherUsageStats",
    "false",
]

with STREAMLIT_LOG.open("w") as log_handle:
    streamlit_process = subprocess.Popen(
        streamlit_command,
        cwd=PROJECT_DIR,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )

STREAMLIT_PID_FILE.write_text(str(streamlit_process.pid))
local_health_url = f"http://127.0.0.1:{PORT}/_stcore/health"

for _ in range(60):
    if streamlit_process.poll() is not None:
        raise RuntimeError(
            "Streamlit stopped during startup. Log tail:\n" + tail(STREAMLIT_LOG)
        )
    try:
        response = requests.get(local_health_url, timeout=2)
        if response.status_code == 200:
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    terminate_process_tree(streamlit_process.pid)
    raise TimeoutError(
        "Streamlit did not become healthy. Log tail:\n" + tail(STREAMLIT_LOG)
    )

print("Streamlit is healthy.")
print("Local URL:", f"http://127.0.0.1:{PORT}")
print("PID:", streamlit_process.pid)

Streamlit is healthy.
Local URL: http://127.0.0.1:8501
PID: 1945


In [25]:
# Cell 6 — Install Cloudflared safely without causing “Text file busy”

import requests

# Set this to True only when you intentionally want to download a fresh binary.
FORCE_CLOUDFLARED_UPDATE = False  # @param {type:"boolean"}

system_name = platform.system().lower()
machine_name = platform.machine().lower()

if machine_name in {"x86_64", "amd64", "x64"}:
    architecture = "amd64"
elif machine_name in {"aarch64", "arm64"}:
    architecture = "arm64"
else:
    raise RuntimeError(f"Unsupported CPU architecture: {platform.machine()}")

if system_name == "linux":
    asset_name = f"cloudflared-linux-{architecture}"
    CLOUDFLARED_PATH = STATE_DIR / "cloudflared"
elif system_name == "windows":
    asset_name = f"cloudflared-windows-{architecture}.exe"
    CLOUDFLARED_PATH = STATE_DIR / "cloudflared.exe"
else:
    raise RuntimeError(
        "This notebook's automatic Cloudflared downloader supports Linux and Windows. "
        "Install Cloudflared with the system package manager on this platform."
    )

cloudflared_url = (
    "https://github.com/cloudflare/cloudflared/releases/latest/download/"
    + asset_name
)


def cloudflared_works(path):
    if not path.exists():
        return False
    try:
        result = subprocess.run(
            [str(path), "version"],
            capture_output=True,
            text=True,
            timeout=15,
        )
        if result.returncode == 0:
            print((result.stdout or result.stderr).strip())
            return True
    except (OSError, subprocess.SubprocessError):
        pass
    return False


# Stop a prior tunnel before any intentional binary replacement.
stop_pid_file(TUNNEL_PID_FILE)

if FORCE_CLOUDFLARED_UPDATE or not cloudflared_works(CLOUDFLARED_PATH):
    # Download to a separate temporary filename. Never use wget -O directly on
    # the executable path because Linux returns “Text file busy” when that file
    # is still mapped by a running process.
    temporary_path = CLOUDFLARED_PATH.with_suffix(
        CLOUDFLARED_PATH.suffix + ".download"
    )
    temporary_path.unlink(missing_ok=True)

    print("Downloading:", cloudflared_url)
    with requests.get(
        cloudflared_url,
        stream=True,
        timeout=120,
        allow_redirects=True,
        headers={"User-Agent": "AIDCE-Sim-launcher/1.0"},
    ) as response:
        response.raise_for_status()
        with temporary_path.open("wb") as destination:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    destination.write(chunk)

    if temporary_path.stat().st_size < 1_000_000:
        raise RuntimeError("Downloaded Cloudflared file is unexpectedly small.")

    if system_name != "windows":
        temporary_path.chmod(0o755)

    # Atomic replacement avoids partially written executables.
    os.replace(temporary_path, CLOUDFLARED_PATH)

if system_name != "windows":
    CLOUDFLARED_PATH.chmod(0o755)

if not cloudflared_works(CLOUDFLARED_PATH):
    raise RuntimeError("Cloudflared installation verification failed.")

print("Cloudflared executable:", CLOUDFLARED_PATH)

cloudflared version 2026.7.3 (built 2026-07-23-09:58 UTC)
cloudflared version 2026.7.3 (built 2026-07-23-09:58 UTC)
Cloudflared executable: /content/.aidce_runtime/cloudflared


In [26]:
# Cell 7 — Create, verify, and print a working Cloudflare Quick Tunnel URL

import requests


def unused_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind(("127.0.0.1", 0))
        return int(sock.getsockname()[1])



def local_streamlit_is_healthy():
    try:
        return requests.get(
            f"http://127.0.0.1:{PORT}/_stcore/health",
            timeout=3,
        ).status_code == 200
    except requests.RequestException:
        return False



def start_quick_tunnel(protocol):
    stop_pid_file(TUNNEL_PID_FILE)

    metrics_port = unused_port()
    metrics_base = f"http://127.0.0.1:{metrics_port}"
    origin_url = f"http://127.0.0.1:{PORT}"

    # Use an explicit empty configuration file so a user's ~/.cloudflared/config.yml
    # cannot interfere with Quick Tunnel mode on a local Linux/Windows system.
    empty_config = STATE_DIR / "cloudflared-empty.yml"
    empty_config.write_text("# intentionally empty\n")

    command = [
        str(CLOUDFLARED_PATH),
        "tunnel",
        "--config",
        str(empty_config),
        "--no-autoupdate",
        "--metrics",
        f"127.0.0.1:{metrics_port}",
        "--protocol",
        protocol,
        "--url",
        origin_url,
    ]

    with TUNNEL_LOG.open("w") as log_handle:
        process = subprocess.Popen(
            command,
            stdout=log_handle,
            stderr=subprocess.STDOUT,
            env={**os.environ, "NO_AUTOUPDATE": "true"},
        )

    TUNNEL_PID_FILE.write_text(str(process.pid))

    hostname = None
    for _ in range(60):
        if process.poll() is not None:
            break

        # Cloudflared exposes the assigned Quick Tunnel hostname here. This is
        # more reliable than depending only on the wording of console logs.
        try:
            response = requests.get(f"{metrics_base}/quicktunnel", timeout=2)
            if response.status_code == 200:
                payload = response.json()
                hostname = payload.get("hostname")
                if hostname:
                    break
        except (requests.RequestException, ValueError):
            pass

        # Retain log parsing only as a fallback.
        log_text = TUNNEL_LOG.read_text(errors="replace") if TUNNEL_LOG.exists() else ""
        matches = re.findall(
            r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
            log_text,
        )
        if matches:
            hostname = matches[-1].removeprefix("https://")
            break

        time.sleep(1)

    if not hostname:
        terminate_process_tree(process.pid)
        return None, "No hostname was issued.\n" + tail(TUNNEL_LOG)

    public_url = f"https://{hostname}"
    health_url = public_url + "/_stcore/health"

    # A newly issued hostname can need several seconds for DNS/edge warm-up.
    last_error = ""
    for _ in range(30):
        if process.poll() is not None:
            last_error = "Cloudflared exited during public URL validation."
            break
        try:
            response = requests.get(health_url, timeout=10, allow_redirects=True)
            if response.status_code == 200:
                return public_url, None
            last_error = f"HTTP {response.status_code} from {health_url}"
        except requests.RequestException as error:
            last_error = str(error)
        time.sleep(2)

    terminate_process_tree(process.pid)
    TUNNEL_PID_FILE.unlink(missing_ok=True)
    return None, last_error + "\n" + tail(TUNNEL_LOG)


if not local_streamlit_is_healthy():
    raise RuntimeError(
        "Streamlit is not healthy. Rerun Cell 5 before creating the tunnel."
    )

# First use Cloudflared's automatic protocol selection. If that does not yield
# a verified URL, retry with HTTP/2, which can work better on restricted networks.
dashboard_url = None
errors = []
for protocol in ("auto", "http2"):
    print(f"Starting Cloudflare Quick Tunnel using protocol={protocol!r} ...")
    dashboard_url, error = start_quick_tunnel(protocol)
    if dashboard_url:
        break
    errors.append(f"[{protocol}] {error}")

if dashboard_url:
    print("\nAIDCE-Sim dashboard is ready:")
    print(dashboard_url)
    print("\nThe URL was checked successfully before being displayed.")
else:
    raise RuntimeError(
        "Cloudflare Quick Tunnel could not be verified.\n\n"
        + "\n\n".join(errors)
        + "\n\nUse Cell 8 as the Colab-session fallback, or use a named "
          "Cloudflare Tunnel/Streamlit Community Cloud for stable hosting."
    )

Starting Cloudflare Quick Tunnel using protocol='auto' ...

AIDCE-Sim dashboard is ready:
https://expanded-framework-usc-plaza.trycloudflare.com

The URL was checked successfully before being displayed.


In [20]:
# Cell 8 — Optional Colab-session fallback when Quick Tunnels are unavailable

# This opens the already-running Streamlit app through Colab's own port proxy.
# It is intended for use inside the current Colab session. It is not a stable,
# permanent deployment URL and may behave differently across Colab versions.

if IS_COLAB:
    from google.colab import output

    output.serve_kernel_port_as_iframe(
        PORT,
        width="100%",
        height=900,
        cache_in_notebook=False,
    )
else:
    print(f"Open http://127.0.0.1:{PORT} in your browser.")

<IPython.core.display.Javascript object>

In [21]:
# Cell 9 — Stop AIDCE-Sim and its tunnel cleanly

stop_pid_file(TUNNEL_PID_FILE)
stop_pid_file(STREAMLIT_PID_FILE)
print("AIDCE-Sim Streamlit server and Cloudflared tunnel have been stopped.")

AIDCE-Sim Streamlit server and Cloudflared tunnel have been stopped.


## Local Linux instructions

Run these commands from the extracted `AIDCE_Sim` directory:

```bash
# Create an isolated Python environment.
python3 -m venv .venv

# Activate it.
source .venv/bin/activate

# Upgrade packaging tools and install the application dependencies.
python -m pip install --upgrade pip setuptools wheel
python -m pip install -r requirements.txt

# Verify the engine.
python smoke_test.py

# Start the dashboard.
python -m streamlit run app.py --server.address 127.0.0.1 --server.port 8501
```

Open `http://127.0.0.1:8501`.

To expose the locally running application temporarily after installing Cloudflared:

```bash
cloudflared tunnel --no-autoupdate --url http://127.0.0.1:8501
```

## Local Windows instructions

Open **Command Prompt** in the extracted `AIDCE_Sim` directory:

```bat
REM Create an isolated Python environment.
py -m venv .venv

REM Activate it.
.venv\Scripts\activate

REM Upgrade packaging tools and install dependencies.
py -m pip install --upgrade pip setuptools wheel
py -m pip install -r requirements.txt

REM Verify the engine.
py smoke_test.py

REM Start the dashboard.
py -m streamlit run app.py --server.address 127.0.0.1 --server.port 8501
```

Open `http://127.0.0.1:8501`.

To expose it temporarily after installing `cloudflared.exe`:

```bat
cloudflared.exe tunnel --no-autoupdate --url http://127.0.0.1:8501
```

## Future TestPyPI installation workflow

After packaging AIDCE-Sim with the import package `aidce_sim` and a console command such as `aidce-sim`, installation should use:

### Linux

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install \
  --index-url https://test.pypi.org/simple/ \
  --extra-index-url https://pypi.org/simple/ \
  aidce-sim

aidce-sim
```

### Windows

```bat
py -m venv .venv
.venv\Scripts\activate
py -m pip install --upgrade pip
py -m pip install --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ aidce-sim

aidce-sim
```

The future package should contain at least:

```text
aidce_sim/
├── __init__.py
├── app.py
├── simulator.py
├── forecasting.py
└── cli.py
```

and should define a console entry point in `pyproject.toml`:

```toml
[project.scripts]
aidce-sim = "aidce_sim.cli:main"
```

The `cli.main()` function should locate the packaged `app.py` and launch it through Streamlit. This separates package installation from temporary Colab tunnelling and gives Linux and Windows users the same command.